# 00. Download the datasets

Downloads the public datasets used to train the UrbanFix category models into `ai_model/data/raw/`. Each cell skips work that is already done, so the notebook can be re-run safely.

| Category | Dataset | Source |
|---|---|---|
| Pothole / Road Damage | Roboflow Pothole Segmentation YOLOv8 (720 train, 60 val) | already in `ai_model/Pothole_Segmentation_YOLOv8.v1i.yolov8/` |
| Garbage / Litter | TACO (1,500 images, COCO masks) | GitHub annotations, images from Flickr and S3 |
| Open Manhole | Road Hazards Dataset (2,716 images, YOLO boxes) | Kaggle `sabidrahman/pothole-cracks-and-openmanhole` |
| Graffiti | STORM graffiti/tagging dataset (1,022 images, boxes) | Zenodo record 3238357 |

In [ ]:
from pathlib import Path

# Works whether the notebook is opened from ai_model/ or ai_model/notebooks/.
ROOT = Path.cwd().resolve()
while ROOT.name != "ai_model" and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert ROOT.name == "ai_model", "Open this notebook from inside the ai_model folder"

DATA = ROOT / "data"          # downloaded and converted datasets (git-ignored)
RAW = DATA / "raw"            # original downloads
RUNS = ROOT / "runs"          # Ultralytics training runs (git-ignored)
WEIGHTS = ROOT / "weights"    # final model files used by the AI service
RESULTS = ROOT / "results"    # metrics used in the project report
for p in (DATA, RAW, RUNS, WEIGHTS, RESULTS):
    p.mkdir(parents=True, exist_ok=True)
print("ai_model root:", ROOT)

In [ ]:
import tarfile, zipfile, json, urllib.request, concurrent.futures as cf


def fetch(url, dest, timeout=600):
    """Download url to dest unless the file is already there."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        print("exists:", dest.name)
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=timeout) as r, open(dest, "wb") as f:
        shutil.copyfileobj(r, f)
    print("downloaded:", dest.name, round(dest.stat().st_size / 1e6, 1), "MB")
    return dest

import shutil

## Graffiti: STORM (Zenodo)

In [ ]:
storm = RAW / "storm"
fetch("https://zenodo.org/api/records/3238357/files/images.tar.gz/content", storm / "images.tar.gz")
fetch("https://zenodo.org/api/records/3238357/files/bounding_boxes.tar.gz/content", storm / "bounding_boxes.tar.gz")
for name in ("images.tar.gz", "bounding_boxes.tar.gz"):
    with tarfile.open(storm / name) as t:
        t.extractall(storm)
print(sorted(p.name for p in storm.iterdir()))

## Garbage / Litter: TACO

The annotation file is on GitHub. Images are fetched from their original Flickr links (640 px versions) and S3 links. Some links are slow or dead; in our run 1,428 of the 1,500 images could be downloaded.

In [ ]:
taco = RAW / "taco"
fetch("https://raw.githubusercontent.com/pedropro/TACO/master/data/annotations.json", taco / "annotations.json")
coco = json.load(open(taco / "annotations.json"))
(taco / "images").mkdir(exist_ok=True)


def taco_path(im):
    return taco / "images" / im["file_name"].replace("/", "_")


def get_image(im):
    out = taco_path(im)
    if out.exists() and out.stat().st_size > 1000:
        return 1
    url = im.get("flickr_640_url") or im.get("flickr_url")
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        data = urllib.request.urlopen(req, timeout=180).read()
        if len(data) > 1000:
            out.write_bytes(data)
            return 1
    except Exception:
        pass
    return 0


# Flickr images are small and fast; S3 originals are large (about 11 MB each), so they get their own pool.
flickr = [im for im in coco["images"] if im.get("flickr_640_url")]
s3 = [im for im in coco["images"] if not im.get("flickr_640_url")]
with cf.ThreadPoolExecutor(48) as ex:
    print("flickr:", sum(ex.map(get_image, flickr)), "of", len(flickr))
with cf.ThreadPoolExecutor(40) as ex:
    print("s3:", sum(ex.map(get_image, s3)), "of", len(s3))
print("TACO images on disk:", len(list((taco / "images").glob("*"))))

## Open Manhole: Road Hazards Dataset (Kaggle)

The Kaggle API serves this public dataset without a login at the time of writing. If it asks for credentials, download the zip from the Kaggle page and place it at `data/raw/road_hazards/road_hazards.zip`.

In [ ]:
rh = RAW / "road_hazards"
zp = fetch("https://www.kaggle.com/api/v1/datasets/download/sabidrahman/pothole-cracks-and-openmanhole",
           rh / "road_hazards.zip", timeout=3600)
with zipfile.ZipFile(zp) as z:
    members = [m for m in z.namelist() if m.startswith(("dataset/dataset/train/", "dataset/dataset/valid/"))]
    z.extractall(rh, members)
for split in ("train", "valid"):
    print(split, len(list((rh / "dataset/dataset" / split / "images").glob("*.jpg"))), "images")